In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

In [2]:
BASE = os.path.join('..', 'data', 'raw')

Paths = {
    'events': os.path.join(BASE, 'floodevents_indofloods.csv'),
    'meta': os.path.join(BASE, 'metadata_indofloods.csv'),
    'catchments': os.path.join(BASE, 'catchment_characteristics_indofloods.csv'),
    'precip': os.path.join(BASE, 'precipitation_variables_indofloods.csv'),
}

In [3]:
events = pd.read_csv(Paths['events'])
meta = pd.read_csv(Paths['meta'])
catchments = pd.read_csv(Paths['catchments'])
precip = pd.read_csv(Paths['precip'])

print("events: ", events.shape)
print("meta: ", meta.shape)
print("catchments: ", catchments.shape)
print("precip: ", precip.shape)

events:  (4548, 13)
meta:  (214, 18)
catchments:  (155, 108)
precip:  (4548, 11)


In [10]:
for name, df in [('events', events), ('meta', meta), ('catchments', catchments), ('precip', precip)]:
    print(f"\n{'='*75}")
    print(f"{name} - {df.shape}")
    print(df.head(2))


events - (4548, 14)
                   EventID Start Date   End Date  Peak Flood Level (m)  \
0  INDOFLOODS-gauge-1010-1 2010-07-21 2010-07-21                 47.95   
1  INDOFLOODS-gauge-1010-2 2016-07-23 2016-07-23                 48.05   

  Peak FL Date  Num Peak FL  Peak Discharge Q (cumec) Peak Discharge Date  \
0   2010-07-21            1                       NaN                 NaN   
1   2016-07-23            1                       NaN                 NaN   

   Flood Volume (cumec)  Event Duration (days)  Time to Peak (days)  \
0                   NaN                      1                    1   
1                   NaN                      1                    1   

   Recession Time (day) Flood Type                GaugeID  
0                     1      Flood  INDOFLOODS-gauge-1010  
1                     1      Flood  INDOFLOODS-gauge-1010  

meta - (214, 18)
                GaugeID  Warning Level  Danger Level      Station  Latitude  \
0  INDOFLOODS-gauge-394         2

In [11]:
# GaugeID is EventID minus the last '-N' suffix
events['GaugeID'] = events['EventID'].str.rsplit('-', n=1).str[0]

# verify it looks right
print(events[['EventID', 'GaugeID']].head(3))

# convert date strings to datetime so we can extract year/month later
for col in ['Start Date', 'End Date', 'Peak FL Date']:
    events[col] = pd.to_datetime(events[col], errors='coerce')

print(events.dtypes[['Start Date', 'End Date', 'Peak FL Date']])

for name, df in [('events', events), ('meta', meta),
                 ('catchments', catchments), ('precip', precip)]:
    null_pct = (df.isnull().sum() / len(df) * 100).round(1)
    null_pct = null_pct[null_pct > 0]  # only show columns that have nulls
    print(f"\n{name} — columns with nulls:")
    print(null_pct.to_string() if len(null_pct) else "  none")

                   EventID                GaugeID
0  INDOFLOODS-gauge-1010-1  INDOFLOODS-gauge-1010
1  INDOFLOODS-gauge-1010-2  INDOFLOODS-gauge-1010
2  INDOFLOODS-gauge-1010-3  INDOFLOODS-gauge-1010
Start Date      datetime64[us]
End Date        datetime64[us]
Peak FL Date    datetime64[us]
dtype: object

events — columns with nulls:
Peak Discharge Q (cumec)    12.2
Peak Discharge Date         12.2
Flood Volume (cumec)        12.7

meta — columns with nulls:
Danger Level             0.9
Basin                    0.9
Source Catchment Area    7.5
Area variation (%)       7.5

catchments — columns with nulls:
No. of Secondorder Streams             2.6
No. of Thirdorder Streams             12.9
No. of Fourthorder Streams            40.0
No. of Fifthorder Streams             60.6
No. of Sixthorder Streams             87.1
No. of Seventhorder Streams           96.8
No. of Eigthorder Streams            100.0
Secondorder Streams Length             3.2
Thirdorder Streams Length             12.9

In [12]:
# these are missing because most catchments don't have high-order streams
# not missing at random — dropping them is correct, not lazy
drop_cols = [c for c in catchments.columns if any(
    x in c for x in ['Fifthorder', 'Sixthorder', 'Seventhorder', 'Eigthorder',
                      'Eighthorder', 'FifthSixth', 'SixthSeventh',
                      'SeventhEighth', 'Sixthseventh']
)]

print(f"Dropping {len(drop_cols)} columns:")
print(drop_cols)
catchment_clean = catchments.drop(columns=drop_cols)
print(f"\ncatchment shape: {catchments.shape} → {catchment_clean.shape}")

Dropping 18 columns:
['No. of Fifthorder Streams', 'No. of Sixthorder Streams', 'No. of Seventhorder Streams', 'No. of Eigthorder Streams', 'Fifthorder Streams Length', 'Sixthorder Streams Length', 'Seventhorder Streams Length', 'Eighthorder Streams Length', 'Fifthorder Streams Mean Length', 'Sixthorder Streams Mean Length', 'Seventhorder Streams Mean Length', 'Eighthorder Streams Mean Length', 'FifthSixth Stream Length Ratio', 'SixthSeventh Stream Length Ratio', 'SeventhEighth Stream Length Ratio', 'FifthSixth Bifurcation Ratio', 'Sixthseventh Bifurcation Ratio', 'SeventhEighth Bifurcation Ratio']

catchment shape: (155, 108) → (155, 90)


In [13]:
# step 1: events + precip (on EventID, 1-to-1)
df = events.merge(precip, on='EventID', how='left')

# step 2: + meta (on GaugeID)
df = df.merge(meta, on='GaugeID', how='left')

# step 3: + catchment (on GaugeID, inner — only keep events that have catchment data)
df = df.merge(catchment_clean, on='GaugeID', how='inner')

print(f"Final shape: {df.shape}")
print(f"Events retained: {len(df)} of 4548")

Final shape: (4548, 130)
Events retained: 4548 of 4548


In [14]:
null_pct = (df.isnull().sum() / len(df) * 100).round(1)
null_pct = null_pct[null_pct > 0]
print(null_pct.to_string())

Peak Discharge Q (cumec)            12.2
Peak Discharge Date                 12.2
Flood Volume (cumec)                12.7
Danger Level                         2.1
Basin                                0.1
Source Catchment Area                1.1
Area variation (%)                   1.1
No. of Secondorder Streams           0.6
No. of Thirdorder Streams            5.5
No. of Fourthorder Streams          28.3
Secondorder Streams Length           0.6
Thirdorder Streams Length            5.5
Fourthorder Streams Length          28.3
Secondorder Streams  Mean Length     0.6
Thirdorder Streams Mean Length       5.5
Fourthorder Streams Mean Length     28.3
FirstSecond Stream Length Ratio      0.6
SecondThird Stream Length Ratio      5.5
ThirdForth Stream Length Ratio      28.3
FourthFifth Stream Length Ratio     57.9
FirstSecond Bifurcation Ratio        0.6
SecondThird Bifurcation Ratio        5.5
ThirdForth Bifurcation Ratio        28.3
FourthFifth Bifurcation Ratio       57.4


In [15]:
# how many unique gauges are in events vs catchment?
print("Unique gauges in events:   ", events['GaugeID'].nunique())
print("Unique gauges in catchment:", catchment_clean['GaugeID'].nunique())

# are all event gauges covered by catchment?
event_gauges = set(events['GaugeID'].unique())
catchment_gauges = set(catchment_clean['GaugeID'].unique())

print("\nEvent gauges NOT in catchment:", len(event_gauges - catchment_gauges))
print("Catchment gauges NOT in events:", len(catchment_gauges - event_gauges))

Unique gauges in events:    155
Unique gauges in catchment: 155

Event gauges NOT in catchment: 0
Catchment gauges NOT in events: 0


In [16]:
# FourthFifth ratio columns are >57% null — unreliable to impute
drop_high_null = [c for c in df.columns
                  if df[c].isnull().mean() > 0.5]

print(f"Dropping {len(drop_high_null)} high-null columns:")
print(drop_high_null)

df = df.drop(columns=drop_high_null)
print(f"\nShape after drop: {df.shape}")

Dropping 2 high-null columns:
['FourthFifth Stream Length Ratio', 'FourthFifth Bifurcation Ratio']

Shape after drop: (4548, 128)


In [17]:
from sklearn.impute import SimpleImputer

# only apply to numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

imputer = SimpleImputer(strategy='median')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

# verify no nulls remain in numeric columns
remaining = df[numeric_cols].isnull().sum().sum()
print(f"Remaining nulls in numeric columns: {remaining}")

Remaining nulls in numeric columns: 0


In [18]:
out_path = os.path.join('..', 'data', 'processed', 'master_clean.csv')
df.to_csv(out_path, index=False)
print(f"Saved to {out_path} — shape: {df.shape}")

Saved to ..\data\processed\master_clean.csv — shape: (4548, 128)
